# Train MWD ConvNet using Google Colaboratory
This files contains code that will let us train MWD CNN models using Google Colab. All you have to do is just run each cell in succession and voila! Under the hood, this program downloads a tarball which contains training, test, and validation data for the city of Aleppo, extracts it into the Colab space and then uses this data to train a CNN to predict tiles that contain buildings that are severely damaged in Aleppo.

## Step 1 - Enter Metadata

Use this section to enter metadata about the model optimization step. This section should be filled in the beginning of evry run. Not changing the RUN_SN will result in old model history files being replaced.

In [ ]:
CITY = 'aleppo' # City of analysis. DON'T change this for the time being
RUN_SN = '1' # Fill In the masterplan document and enter SN as per the Masterplan Document (https://docs.google.com/document/d/1mrrPWecoxVaoPa2RkIyZ0QhpMUDbVxJe02t1yv-8B44/edit?usp=sharing)
USER = 'AROGYA' # Replace with your name


## Step 2 - Dependencies and Imports

This section loads required files, libraries and scripts necessary for our analysis. First, we download the training (balanced), validation and test data set *.tgz file from google drive, and extract contents into project directory

In [11]:
!gdown 1NTUes3NS-sSiJWVjY_90noWvHJiidRzJ

Access denied with the following error:

 	Too many users have viewed or downloaded this file recently. Please
	try accessing the file again later. If the file you are trying to
	access is particularly large or is shared with many people, it may
	take up to 24 hours to be able to view or download the file. If you
	still can't access a file after 24 hours, contact your domain
	administrator. 

You may still be able to access the file from the browser:

	 https://drive.google.com/uc?id=1NTUes3NS-sSiJWVjY_90noWvHJiidRzJ 



In [ ]:
!tar -xvf data.tgz
!rm -rf data.tgz

Streaming output truncated to the last 5000 lines.
data/aleppo_images_train.zarr/0.3.0.2
data/aleppo_images_train.zarr/13.7.4.1
data/aleppo_images_train.zarr/34.6.7.1
data/aleppo_images_train.zarr/39.0.7.2
data/aleppo_images_train.zarr/3.0.1.0
data/aleppo_images_train.zarr/16.0.3.0
data/aleppo_images_train.zarr/31.1.0.0
data/aleppo_images_train.zarr/18.5.2.1
data/aleppo_images_train.zarr/32.2.1.2
data/aleppo_images_train.zarr/15.3.2.2
data/aleppo_images_train.zarr/8.2.7.0
data/aleppo_images_train.zarr/6.7.6.1
data/aleppo_images_train.zarr/17.2.0.0
data/aleppo_images_train.zarr/30.3.3.0
data/aleppo_images_train.zarr/19.7.1.1
data/aleppo_images_train.zarr/33.0.2.2
data/aleppo_images_train.zarr/14.1.1.2
data/aleppo_images_train.zarr/9.0.4.0
data/aleppo_images_train.zarr/7.5.5.1
data/aleppo_images_train.zarr/1.1.3.2
data/aleppo_images_train.zarr/12.5.7.1
data/aleppo_images_train.zarr/35.4.4.1
data/aleppo_images_train.zarr/38.2.4.2
data/aleppo_images_train.zarr/2.2.2.0
data/aleppo_images_tr

Next we clone the latest repo so that we have all the required scripts at our disposal.

In [ ]:
!git clone https://github.com/goclem/destruction.git
%cd destruction
!git checkout ak-exploration

Cloning into 'destruction'...
remote: Enumerating objects: 333, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 333 (delta 22), reused 28 (delta 13), pack-reused 294
Receiving objects: 100% (333/333), 1.92 MiB | 666.00 KiB/s, done.
Resolving deltas: 100% (195/195), done.
/Users/arogyak/projects/mwd/destruction/notebooks/destruction
Branch 'ak-exploration' set up to track remote branch 'ak-exploration' from 'origin'.
Switched to a new branch 'ak-exploration'


Next we install required libraries and import all of our dependencies

In [ ]:
!pip install geopandas
!pip install rasterio
!pip install zarr

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 1.0 MB 5.1 MB/s 
     |████████████████████████████████| 6.3 MB 35.9 MB/s 
     |████████████████████████████████| 16.7 MB 380 kB/s 
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 19.3 MB 5.4 MB/s 
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 185 kB 5.5 MB/s 
     |████████████████████████████████| 6.6 MB 34.5 MB/s 
  Created wheel for asciitree: filename=asciitree-0.3.3-py3-none-any.whl size=5050 sha256=bb60e46c8423c6dfca00457472a3ce49f21fcf34b27b3c0df31589ba6be17aca
  Stored in directory: /root/.cache/pip/wheels/12/1c/38/0def51e15add93bff3f4bf9c248b94db0839b980b8535e72a0
Successfully built asciitree


In [ ]:
# Modules
import numpy as np
import pandas as pd
import itertools
import destruction_models as models
import tensorflow as tf
import random

from destruction_utilities import *
from destruction_statistics import *
from numpy import random
import matplotlib.pyplot as plt
from tensorflow.keras import callbacks, preprocessing
from tensorflow.keras.utils import Sequence
from tensorflow.keras.metrics import CategoricalAccuracy, Precision, AUC
from tensorflow.keras.models import load_model
from sklearn.metrics import precision_recall_curve, roc_auc_score
from os import path
import zarr
import shutil
from tensorflow.keras import backend as K
import gc
import time
import pickle

## Step 3 - Functions and classes
Next we write some helper functions and classes to make our life easier.

In [ ]:
def get_zarr(city, type, set, balanced = False):
    if balanced:
        path = f'../data/{city}_{type}s_{set}_balanced.zarr'
    else:
        path = f'../data/{city}_{type}s_{set}.zarr'
    return zarr.open(path, mode='r')


def get_path(city, type, set, balanced = False):
    if balanced:
        path = f'../data/{city}_{type}s_{set}_balanced.zarr'
    else:
        path = f'../data/{city}_{type}s_{set}.zarr'
    return path

def reshuffle_data(X, y): 
    print("Shuffling data started...")
    
    balanced_path_images = get_path(CITY, 'image', 'train', balanced=True)
    balanced_path_labels = get_path(CITY, 'label', 'train', balanced=True)
    shuffled_path_images = get_path(CITY, 'image', 'train', balanced=True).split('.zarr')[0] + "_shuffled.zarr"
    shuffled_path_labels = get_path(CITY, 'label', 'train', balanced=True).split('.zarr')[0] + "_shuffled.zarr"
    
    
    shutil.rmtree(shuffled_path_images, ignore_errors=True)
    shutil.rmtree(shuffled_path_labels, ignore_errors=True)
    n=X.shape[0]
    
    tuple_pair = make_tuple_pair(n, 250)
    
    np.random.shuffle(tuple_pair)
    
    zarr.save(shuffled_path_images, np.empty((0,*TILE_SIZE,3)))
    zarr.save(shuffled_path_labels, np.empty((0,1,1,1)))
    
    X1 = zarr.open(shuffled_path_images, mode='a')
    y1 = zarr.open(shuffled_path_labels, mode='a')
    
    print(f"Reordering array in batches of 250. Total {len(tuple_pair)} sets..")
    for i, t in enumerate(tuple_pair):
        if i % 50 == 0 and i!=0:
            print(f"Finished {i} sets")
        Xs = X[t[0]:t[1]]
        ys = y[t[0]:t[1]]
        X1.append(Xs)
        y1.append(ys)
    
    
    shutil.rmtree(balanced_path_images)
    shutil.rmtree(balanced_path_labels)
    
    zarr.save(balanced_path_images, np.empty((0,*TILE_SIZE,3)))
    zarr.save(balanced_path_labels, np.empty((0,1,1,1)))
    
    X_shuffled = zarr.open(balanced_path_images, mode='a')
    y_shuffled = zarr.open(balanced_path_labels, mode='a')
    
    tuple_pair = make_tuple_pair(n, 10000)
    print(f"Shuffling array in batches of 10000. Total {len(tuple_pair)} sets..")
    for i, t in enumerate(tuple_pair):
        if i % 5 == 0 and i != 0:
            print(f"Finished {i} sets")
        shuffled = np.arange(0, 10000)
        np.random.shuffle(shuffled)
        Xs = X1[t[0]:t[1]][shuffled]
        ys = y1[t[0]:t[1]][shuffled]
        X_shuffled.append(Xs)
        y_shuffled.append(ys)
        
    shutil.rmtree(shuffled_path_images)
    shutil.rmtree(shuffled_path_labels)
    return X_shuffled, y_shuffled

def recall_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    recall = true_positives / (possible_positives + K.epsilon())
    return recall

def precision_m(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    return precision

def f1_m(y_true, y_pred):
    precision = precision_m(y_true, y_pred)
    recall = recall_m(y_true, y_pred)
    return 2*((precision*recall)/(precision+recall+K.epsilon()))

auc = AUC(
    num_thresholds=200,
    curve='ROC',
)
    
def run_model(training_images, training_labels, validation_images, validation_labels, epochs=50):
    training_generator = ZarrGenerator(training_images, training_labels, batch_size=32)
    validation_generator = ZarrGenerator(validation_images, validation_labels, batch_size=32)
    
    training_callbacks = [
        callbacks.EarlyStopping(monitor='val_auc', restore_best_weights=True, patience=5),
    #     callbacks.BackupAndRestore(backup_dir='../models')
    ]

    args  = dict(shape=(128, 128, 3), filters=16, units=32, dropout=0.1) # ! Check parameters before run
    model = models.convolutional_network(**args)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', f1_m, precision_m, recall_m, auc ])

    # Train model on dataset
    history = model.fit_generator(
        training_generator,
        validation_data=validation_generator, 
        epochs=epochs, 
        callbacks = training_callbacks
    )
    
    return model, history


def calculate_auc(model, test_images, test_labels):
    gc.collect(generation=2)    
    batch_size = 5000
    iters = test_images.shape[0] // batch_size
    preds = []
    labels = []
    for i in range(0, iters):
        end = (i+1)*batch_size
        if i == iters - 1:
            preds.append(model.predict(test_images[i*batch_size:]))
            labels.append(test_labels[i*batch_size:])
        else:
            preds.append(model.predict(test_images[i*batch_size: end]))
            labels.append(test_labels[i*batch_size: end])
            
    yhat = np.squeeze(np.concatenate(preds, axis=0))
    y = np.squeeze(np.concatenate(labels, axis=0 ))
    roc_auc = roc_auc_score(y, yhat)
    
    return roc_auc

class ZarrGenerator(Sequence):
    def __init__(self, images, labels, batch_size=32):
        self.images = images
        self.labels = labels
        self.batch_size = batch_size
        
    def __len__(self):
        return len(self.images)//self.batch_size
    
    def __getitem__(self, index):

        X = self.images[index*self.batch_size:(index+1)*self.batch_size]
        y = self.labels[index*self.batch_size:(index+1)*self.batch_size]
        
        return self.augment(X), y.flatten()
    
    def augment(self, X):
        # Horizontal and vertical flip
        flipping_funcs = [
            lambda image: image,
            lambda image: np.fliplr(image),
            lambda image: np.flipud(image),
            lambda image: np.flipud(np.fliplr(image))
        ]
        func = random.choice(flipping_funcs)
        X = func(X)
        
        # Brightness
        alpha = random.choice(np.linspace(0.85, 1.4))
        X = X * alpha
        
        return X
    

In [ ]:
training_images = get_zarr(CITY, 'image', 'train', balanced=True)
training_labels = get_zarr(CITY, 'label', 'train', balanced=True)
validation_images = get_zarr(CITY, 'image', 'validate')
validation_labels = get_zarr(CITY, 'label', 'validate')
test_images = get_zarr(CITY, 'image', 'test')
test_labels = get_zarr(CITY, 'label', 'test')

../data/aleppo_images_train_balanced.zarr
../data/aleppo_labels_train_balanced.zarr
../data/aleppo_images_validate.zarr
../data/aleppo_labels_validate.zarr
../data/aleppo_images_test.zarr
../data/aleppo_labels_test.zarr


## Step 4 - Run Experiment

This is where the magic happens, a for-loop that, in every loop: 

* shuffles our dataset, 
* fits a model (max 50 epochs), 
* stores model metrics for every run, and
* saves models with test_auc > 0.8



In [ ]:
for i in range(0,50):
    training_images_shuffled, training_labels_shuffled = reshuffle_data(training_images, training_labels)
    print("Shuffling complete..")
    model = run_model(training_images_shuffled, training_labels_shuffled, validation_images, validation_labels, epochs=50)
    history = model[1]
    model = model[0]
    print("Model optimization complete..")
    auc = calculate_auc(model, test_images, test_labels)
    
    with open(f'../models/{USER}_SN{RUN_SN}_RUN{i}_{CITY}_{ts}_hist', 'wb') as file_pi:
        pickle.dump(history.history, file_pi)
    
    if(auc > 0.8):
        ts = str(time.time())
        model.save(f'../models/{USER}_SN{RUN_SN}_RUN{i}_{CITY}_{ts}', save_format="h5")

Shuffling complete..


/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:115: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.


Epoch 1/50
2352/7500 [========>.....................] - ETA: 1:00:08 - loss: 0.6791 - accuracy: 0.6000 - f1_m: 0.6059 - precision_m: 0.5939 - recall_m: 0.6378 - auc: 0.6271